# 01 — Explore & quality-check openFDA drug/event (bronze)

Step 1 of the pipeline: **look** at the raw data and run the 5 data-quality checks
(Completeness, Accuracy, Consistency, Timeliness, Uniqueness) — the week-12 pattern.
This only reads the data; it changes nothing.

In [11]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('openfda_explore')
         .config('spark.driver.memory', '4g')
         .getOrCreate())
spark

In [12]:
# Read all days. For a FAST first run, point at one day, e.g. receivedate=20240102
DATA_GLOB = '/home/jovyan/data/bronze/drug_event/receivedate=*/*.json'
df = spark.read.json(DATA_GLOB)
total = df.count()
print('Total rows:', total)

Total rows: 2687675


### Why cache the data as Parquet?

The first read was raw **nested JSON** (3,043 files). Every check re-read and **re-parsed** all
of it from scratch — one completeness cell took **over an hour**. JSON is great for *landing* raw
data, but slow to *work* with.

So I save it once as **Parquet**, then read from that. Parquet is much faster because it is:

- **Binary with the schema stored inside** → no text-parsing or schema-inference on every read
  (JSON has to be parsed every single time).
- **Columnar** → a query reads only the columns it needs, not every field of every record.
- **Compressed** → smaller on disk, so less to read.
- **Fewer, bigger files** → avoids the overhead of opening thousands of small JSON parts.

Result: reads drop from **minutes to seconds**. Classic pattern — **JSON to land, Parquet to
work** — and it's the start of the Silver layer.

In [13]:
%%time
# One-time: read the slow nested JSON once and save it as Parquet (fast columnar format).
df.write.mode('overwrite').parquet('/home/jovyan/dq_cache/drug_event')

CPU times: user 393 ms, sys: 536 ms, total: 929 ms
Wall time: 15min 53s


In [14]:
%%time
# Point df at the Parquet cache — every cell after this reads it in seconds, not minutes.
df = spark.read.parquet('/home/jovyan/dq_cache/drug_event')
total = df.count()
print('rows:', total)

rows: 2687675
CPU times: user 54.1 ms, sys: 6.71 ms, total: 60.8 ms
Wall time: 1.98 s


## 1. Structure — what fields, and how nested?

In [15]:
df.printSchema()

root
 |-- authoritynumb: string (nullable = true)
 |-- companynumb: string (nullable = true)
 |-- duplicate: string (nullable = true)
 |-- fulfillexpeditecriteria: string (nullable = true)
 |-- occurcountry: string (nullable = true)
 |-- patient: struct (nullable = true)
 |    |-- drug: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- actiondrug: string (nullable = true)
 |    |    |    |-- activesubstance: struct (nullable = true)
 |    |    |    |    |-- activesubstancename: string (nullable = true)
 |    |    |    |-- drugadditional: string (nullable = true)
 |    |    |    |-- drugadministrationroute: string (nullable = true)
 |    |    |    |-- drugauthorizationnumb: string (nullable = true)
 |    |    |    |-- drugbatchnumb: string (nullable = true)
 |    |    |    |-- drugcharacterization: string (nullable = true)
 |    |    |    |-- drugcumulativedosagenumb: string (nullable = true)
 |    |    |    |-- drugcumulativedosageunit: st

### Schema — findings
- **Every field is `string`** — even dates (`receivedate`) and codes (`serious`, `patientsex`)
  → **cast** them in the Silver layer.
- **Two nested arrays to flatten:** `patient.drug[]` (with an `openfda` block inside) and
  `patient.reaction[]` → exploding to one row per (report, drug, reaction) is the core Silver
  step (#4 / #5).
- **Matches the field dictionary** ✅ — no surprise fields.
- Note: `reportduplicate` shows as `string` (spec says object) — a Spark inference quirk; that
  field is rarely filled.
- Full schema saved in `docs/Metadata/drug_event_schema.md`.r next commit.)

## 2. Uniqueness — is each `safetyreportid` one row?  →  feeds #4 dedup

Why safetyreportid for uniqueness? it's the core ID.

The metadata says it plainly: it's the case / report ID (NNNNNNN-C — the 7 digits are the report, the last digit is a checksum).

 One report = one safetyreportid. It's the natural primary key at the report grain, and it's the degenerate ID on your future fact table. So checking it is correct — it is the core id.

In [17]:
distinct_ids = df.select('safetyreportid').distinct().count()
print('rows:', total)
print('distinct safetyreportid:', distinct_ids)
print('duplicate rows:', total - distinct_ids)

dups = df.groupBy('safetyreportid').count().filter('count > 1')
print('safetyreportids appearing more than once:', dups.count())
dups.orderBy(F.desc('count')).show(5)

rows: 2687675
distinct safetyreportid: 2687675
duplicate rows: 0
safetyreportids appearing more than once: 0
+--------------+-----+
|safetyreportid|count|
+--------------+-----+
+--------------+-----+



### Uniqueness — findings
- **0 duplicate `safetyreportid`** (2,687,675 unique = total rows).
- openFDA returns latest-version-only → **report-level dedup already done by the source**.
- Still to check: duplicates at the **(report, drug, reaction)** grain after exploding.

## 3. Completeness — how often are key fields missing?  →  feeds #6

In [18]:
%%time
for c in ['safetyreportid', 'safetyreportversion', 'receivedate', 'serious', 'patient']:
    n = df.filter(F.col(c).isNull()).count()
    print(f'{c:20s} {100*n/total:6.2f}% null')

safetyreportid         0.00% null
safetyreportversion    0.00% null
receivedate            0.00% null
serious                0.01% null
patient                0.00% null
CPU times: user 99.3 ms, sys: 65.4 ms, total: 165 ms
Wall time: 1min 34s


For below cell: add the blank check, and extend it to the fields that **actually matter**. Two reasons:
- `isNull()` misses **empty strings** → check null **OR** blank.
- Your check only tested **top-level** fields. `patient` showing "0% null" just means the struct *exists* — it hides whether `patient.patientsex`, the `drug`/`reaction` arrays, and the **drug name** inside are actually filled.

Your current numbers are a good sign (key fields ~0% null; `serious` 0.01% ≈ ~270 rows). Now go deeper

In [19]:
%%time
#null-or-blank, including nested fields + arrays:
from pyspark.sql import functions as F

def missing_pct(frame, col, denom):
    """% of rows where a string column is null OR blank/whitespace."""
    n = frame.filter(F.col(col).isNull() | (F.trim(F.col(col)) == '')).count()
    return 100 * n / denom

# key report + patient fields — null OR blank (includes nested)
fields = ['safetyreportid', 'safetyreportversion', 'receivedate', 'serious',
          'patient.patientsex', 'patient.patientonsetage',
          'primarysource.qualification', 'occurcountry']
print('--- null-or-blank % ---')
for c in fields:
    print(f'{c:30s} {missing_pct(df, c, total):6.2f}%')

# arrays: reports with NO drugs / NO reactions
print('\n--- empty or null arrays ---')
for arr in ['patient.drug', 'patient.reaction']:
    n = df.filter(F.col(arr).isNull() | (F.size(F.col(arr)) == 0)).count()
    print(f'{arr:30s} {100*n/total:6.2f}% empty/null')

--- null-or-blank % ---
safetyreportid                   0.00%
safetyreportversion              0.00%
receivedate                      0.00%
serious                          0.01%
patient.patientsex              16.84%
patient.patientonsetage         43.73%
primarysource.qualification      0.97%
occurcountry                    11.20%

--- empty or null arrays ---
patient.drug                     0.00% empty/null
patient.reaction                 0.00% empty/null
CPU times: user 132 ms, sys: 85.3 ms, total: 217 ms
Wall time: 1min 32s


In [20]:
%%time
#the atomic fields cleaning actually uses (drug name, reaction):
drugs = df.select(F.explode(F.col('patient.drug')).alias('d')).select(F.col('d.medicinalproduct').alias('medicinalproduct'))
nd = drugs.count()
print(f'medicinalproduct null-or-blank: {missing_pct(drugs, "medicinalproduct", nd):.2f}%  ({nd:,} drug rows)')

rx = df.select(F.explode(F.col('patient.reaction')).alias('r')).select(F.col('r.reactionmeddrapt').alias('reactionmeddrapt'))
nr = rx.count()
print(f'reactionmeddrapt null-or-blank: {missing_pct(rx, "reactionmeddrapt", nr):.2f}%  ({nr:,} reaction rows)')

medicinalproduct null-or-blank: 0.00%  (10,367,170 drug rows)
reactionmeddrapt null-or-blank: 0.00%  (8,025,550 reaction rows)
CPU times: user 93.1 ms, sys: 39.2 ms, total: 132 ms
Wall time: 1min 26s


In [21]:
%%time
drugs = df.select(F.explode(F.col('patient.drug')).alias('d')).select(F.col('d.medicinalproduct').alias('name'))
print('distinct drug names:', drugs.select('name').distinct().count())
drugs.groupBy('name').count().orderBy(F.desc('count')).show(25, truncate=False)

distinct drug names: 97789
+----------------+------+
|name            |count |
+----------------+------+
|DUPIXENT        |201683|
|MOUNJARO        |172011|
|PREDNISONE      |129250|
|METHOTREXATE    |118126|
|RITUXIMAB       |82697 |
|HUMIRA          |79006 |
|ACETAMINOPHEN   |74381 |
|REPATHA         |68147 |
|ACTEMRA         |64910 |
|ASPIRIN         |64536 |
|DEXAMETHASONE   |63427 |
|INFLECTRA       |57416 |
|FOLIC ACID      |56308 |
|ZEPBOUND        |53186 |
|VEDOLIZUMAB     |51664 |
|SULFASALAZINE   |48530 |
|ATORVASTATIN    |48247 |
|ELIQUIS         |46109 |
|REVLIMID        |45649 |
|CYCLOPHOSPHAMIDE|45396 |
|COSENTYX        |44933 |
|GABAPENTIN      |44843 |
|INFLIXIMAB      |44525 |
|PREDNISOLONE    |42447 |
|OMEPRAZOLE      |42356 |
+----------------+------+
only showing top 25 rows

CPU times: user 20.4 ms, sys: 10.2 ms, total: 30.5 ms
Wall time: 10.2 s


In [22]:
%%time
drugs2 = df.select(F.explode(F.col('patient.drug')).alias('d'))
nd = drugs2.count()
for col in ['d.openfda.generic_name', 'd.openfda.substance_name', 'd.openfda.rxcui']:
    n = drugs2.filter(F.col(col).isNull() | (F.size(F.col(col)) == 0)).count()
    print(f'{col:30s} {100*n/nd:6.2f}% missing')

d.openfda.generic_name          16.47% missing
d.openfda.substance_name        18.40% missing
d.openfda.rxcui                 18.70% missing
CPU times: user 331 ms, sys: 160 ms, total: 491 ms
Wall time: 4min 34s


### Completeness — findings

**Required fields are complete** (no nulls, no blanks):
- `safetyreportid` (unique + complete), `receivedate`, `serious` (0.01%)
- `medicinalproduct` (drug name) — 0% missing across 10.4M drug rows
- `reactionmeddrapt` (reaction) — 0% missing across 8.0M reaction rows
- Every report has ≥1 drug and ≥1 reaction (arrays 0% empty)

**Demographics are partial** → bucket as `Unknown` (doesn't affect signals):
- `patientonsetage` 44%, `patientsex` 17%, `occurcountry` 11% missing

**No hidden empty strings** — the null-or-blank check matched null-only.

**Drug names (feeds #5 normalisation):**
- 97,789 distinct names; top ones are clean, the mess is the long tail (brand vs generic, misspellings).
- openFDA already resolves most: `openfda.generic_name` present ~83%, `substance_name` ~82%, `rxcui` ~81%.
- → #5 plan: **Tier 1** use `openfda.generic_name`/`substance_name`; **Tier 2** clean/flag the ~17% leftover, and publish the resolution rate.

**Fan-out:** ~3.9 drugs and ~3.0 reactions per report.

## 4. Consistency — do coded fields hold only valid values?  →  feeds #6
`serious` should be 1 or 2; `patientsex` should be 0, 1 or 2.

In [24]:
%%time
# serious: 1=serious, 2=not serious
print('=== serious ===')
df.groupBy('serious').count().orderBy('serious').show()

# patientsex: 0=Unknown, 1=Male, 2=Female
print('=== patientsex ===')
df.select(F.col('patient.patientsex').alias('patientsex')).groupBy('patientsex').count().orderBy('patientsex').show()

# drugcharacterization: 1=Suspect, 2=Concomitant, 3=Interacting (used by signal metrics)
print('=== drugcharacterization ===')
df.select(F.explode(F.col('patient.drug')).alias('d')) \
  .groupBy(F.col('d.drugcharacterization').alias('drugcharacterization')).count().orderBy('drugcharacterization').show()

# reactionoutcome: 1=Recovered, 2=Recovering, 3=Not recovered, 4=Recovered w/ sequelae, 5=Fatal, 6=Unknown
print('=== reactionoutcome ===')
df.select(F.explode(F.col('patient.reaction')).alias('r')) \
  .groupBy(F.col('r.reactionoutcome').alias('reactionoutcome')).count().orderBy('reactionoutcome').show()

=== serious ===
+-------+-------+
|serious|  count|
+-------+-------+
|   NULL|    161|
|      1|1461759|
|      2|1225755|
+-------+-------+

=== patientsex ===
+----------+-------+
|patientsex|  count|
+----------+-------+
|      NULL| 452532|
|         0|  12078|
|         1| 908201|
|         2|1314864|
+----------+-------+

=== drugcharacterization ===
+--------------------+-------+
|drugcharacterization|  count|
+--------------------+-------+
|                NULL|      1|
|                   1|6611113|
|                   2|3689221|
|                   3|  66817|
|                   4|     17|
|                   5|      1|
+--------------------+-------+

=== reactionoutcome ===
+---------------+-------+
|reactionoutcome|  count|
+---------------+-------+
|           NULL| 359868|
|              1|1051443|
|              2| 537305|
|              3|1095999|
|              4|  26129|
|              5| 473401|
|              6|4481405|
+---------------+-------+

CPU times: user 37

### Consistency — findings

Coded fields checked against their valid code sets:
- `serious` (1/2), `patientsex` (0/1/2), `reactionoutcome` (1–6) — **all clean** (only valid codes + nulls).
- `drugcharacterization` (should be 1/2/3) — **18 rogue rows**: 17× `4`, 1× `5` (0.0002% of 10.4M).
  → validation rule for #6: `drugcharacterization ∈ {1,2,3}`, quarantine violators.

Distributions: ~64% Suspect / ~36% Concomitant drugs; reaction outcome mostly Unknown (`6`, ~56%).
No rollup/aggregate rows (every row = one report), so grain-consistency is N/A.

## 5. Timeliness — oldest and newest `receivedate`

In [25]:
df.select(F.min('receivedate').alias('oldest'), F.max('receivedate').alias('newest')).show()

+--------+--------+
|  oldest|  newest|
+--------+--------+
|20230101|20241231|
+--------+--------+



### Timeliness — findings
- receivedate spans exactly 2023-01-01 → 2024-12-31 (min/max = START/END, so ingestion bounds are
  correct — no leakage, all valid YYYYMMDD).
- Fixed historical window by design; openFDA is batch/quarterly, not real-time.
  "Freshness" = when ingestion is last re-run.

## 6. Drug-name messiness — top `medicinalproduct` values  →  feeds #5 normalisation
One report has many drugs, so we `explode` `patient.drug` into one row per drug first.

In [26]:
drugs = df.select(F.explode('patient.drug').alias('d'))
print('total drug rows:', drugs.count())
drugs.select('d.medicinalproduct').groupBy('medicinalproduct').count().orderBy(F.desc('count')).show(20, truncate=False)

total drug rows: 10367170
+----------------+------+
|medicinalproduct|count |
+----------------+------+
|DUPIXENT        |201683|
|MOUNJARO        |172011|
|PREDNISONE      |129250|
|METHOTREXATE    |118126|
|RITUXIMAB       |82697 |
|HUMIRA          |79006 |
|ACETAMINOPHEN   |74381 |
|REPATHA         |68147 |
|ACTEMRA         |64910 |
|ASPIRIN         |64536 |
|DEXAMETHASONE   |63427 |
|INFLECTRA       |57416 |
|FOLIC ACID      |56308 |
|ZEPBOUND        |53186 |
|VEDOLIZUMAB     |51664 |
|SULFASALAZINE   |48530 |
|ATORVASTATIN    |48247 |
|ELIQUIS         |46109 |
|REVLIMID        |45649 |
|CYCLOPHOSPHAMIDE|45396 |
+----------------+------+
only showing top 20 rows



### Drug names — findings
- 10,367,170 drug rows; 97,789 distinct `medicinalproduct` values.
- Top names clean/recognisable; mess is the long tail (brand vs generic, misspellings, combos).
- openFDA resolves ~83% (`openfda.generic_name`).
- → #5: Tier-1 `openfda.generic_name`/`substance_name`; Tier-2 clean/flag the ~17% tail; publish resolution rate.

## 7. Fan-out — how many drugs & reactions per report?

In [27]:
df.select(F.size('patient.drug').alias('n_drugs'),
          F.size('patient.reaction').alias('n_reactions')).describe().show()

+-------+------------------+------------------+
|summary|           n_drugs|       n_reactions|
+-------+------------------+------------------+
|  count|           2687675|           2687675|
|   mean|3.8573004548541023| 2.986056721887877|
| stddev|13.781278622017565|4.4667582892794195|
|    min|                 1|                 1|
|    max|              4113|               518|
+-------+------------------+------------------+



### Fan-out — findings
- Drugs per report: mean 3.86, min 1, **max 4,113**.
- Reactions per report: mean 2.99, min 1, **max 518**.
- Usually small, but extreme outliers exist (high stddev).
- Implication: crossing drug × reaction can explode on mega-reports (4113 × 518 ≈ 2.1M pairs from
  one report) → data skew to handle in Silver (cap it, or at least watch for it).

## Summary — what exploration found, and what it means for cleaning

| Dimension | Result | Cleaning action |
|---|---|---|
| **Uniqueness** | 0 duplicate `safetyreportid` (2.69M unique) | #4: report-level dedup already done by openFDA; re-check the (report, drug, reaction) grain after flatten |
| **Completeness** | key fields 100% (id, dates, drug name, reaction); demographics partial (age 44%, sex 17%, country 11%) | #6: demographics → `Unknown` bucket |
| **Consistency** | coded fields valid, except **18 rogue `drugcharacterization`** (codes 4/5) | #6: quarantine `drugcharacterization ∉ {1,2,3}` |
| **Drug names (#5)** | 97,789 distinct names; ~83% resolved by `openfda.generic_name` | #5: Tier-1 `openfda.generic_name`/`substance_name`; Tier-2 clean/flag the ~17% tail |
| **Fan-out / skew** | ~3.9 drugs & ~3.0 reactions per report, but max **4,113 drugs** / **518 reactions** | Silver: handle mega-report skew when flattening |
| **Timeliness** | exact 2023–2024 window; batch/quarterly source | note only |

These findings define the cleaning work: **#4 dedup · #5 normalisation · #6 validation**.